# Baseline delay model on the joint table

Run after the flight–weather joint table has been created. This is the simple downstream validation model described in the report, not a claim to reproduce the referenced paper.


In [ ]:


from pathlib import Path
from pyspark.sql import SparkSession, functions as F
from functools import reduce

from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

# =========================================================
# 1. Spark session
# =========================================================
PROJECT_ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents) if (p / "notebooks/project_clean.ipynb").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Open this notebook from inside the cloned project directory.")
DATA_ROOT = PROJECT_ROOT / "data"

spark = (
    SparkSession.builder
    .appName("flight_delay_rf_simple")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

# =========================================================
# 2. Read JT parquet
# =========================================================
JT_PATH = str(DATA_ROOT / "jt")

months_to_merge = [
    (2012, 1),
    (2012, 5),
    (2012, 7),
    (2012, 9),
    (2013, 4),
    (2013, 6),
    (2013, 9),
    (2013, 11),
]

dfs = []
for y, m in months_to_merge:
    path = f"{JT_PATH}/year={y}/month={m}"
    df = (
        spark.read.parquet(path)
        .withColumn("jt_year", F.lit(y))
        .withColumn("jt_month", F.lit(m))
    )
    dfs.append(df)

jt_all = reduce(lambda a, b: a.unionByName(b, allowMissingColumns=True), dfs)

print("Total rows:", jt_all.count())
print("Total columns:", len(jt_all.columns))
print("First 20 columns:", jt_all.columns[:20])

jt_all.groupBy("jt_year", "jt_month").count().orderBy("jt_year", "jt_month").show()

# =========================================================
# 3. Basic feature engineering
# =========================================================
jt_ml = (
    jt_all
    .withColumn("scheduled_dep_ts", F.to_timestamp("scheduled_dep_ts"))
    .withColumn("dep_year", F.year("scheduled_dep_ts"))
    .withColumn("dep_month", F.month("scheduled_dep_ts"))
    .withColumn("dep_day", F.dayofmonth("scheduled_dep_ts"))
    .withColumn("dep_dayofweek", F.dayofweek("scheduled_dep_ts"))
    .withColumn("dep_hour", F.hour("scheduled_dep_ts"))
    .withColumn("dep_minute", F.minute("scheduled_dep_ts"))
    .withColumn("is_weekend", F.when(F.dayofweek("scheduled_dep_ts").isin([1, 7]), 1).otherwise(0))
)

target_col = "is_delayed_15"

base_feature_cols = [
    "OP_CARRIER_AIRLINE_ID",
    "OP_CARRIER_FL_NUM",
    "ORIGIN_AIRPORT_ID",
    "DEST_AIRPORT_ID",
    "flight_yyyymm",
    "jt_month",
    "dep_year",
    "dep_month",
    "dep_day",
    "dep_dayofweek",
    "dep_hour",
    "dep_minute",
    "is_weekend",
]

weather_num_patterns = [
    "_Visibility",
    "_DryBulbFarenheit",
    "_WetBulbFarenheit",
    "_DewPointFarenheit",
    "_RelativeHumidity",
    "_WindSpeed",
    "_WindDirection",
    "_StationPressure",
    "_SeaLevelPressure",
    "_HourlyPrecip",
    "_Altimeter",
    "time_diff_sec",
]

weather_feature_cols = [
    c for c in jt_ml.columns
    if any(p in c for p in weather_num_patterns)
]

feature_cols = base_feature_cols + weather_feature_cols

print("Number of weather features:", len(weather_feature_cols))
print("Total number of features:", len(feature_cols))
print("Sample weather features:", weather_feature_cols[:20])

jt_model = jt_ml.select(feature_cols + [target_col, "jt_year"])

print("Model rows:", jt_model.count())
print("Model cols:", len(jt_model.columns))
jt_model.show(3, truncate=False)

jt_model.groupBy(target_col).count().orderBy(target_col).show()

# =========================================================
# 4. Cast all feature cols + label to double
# =========================================================

for c in feature_cols + [target_col]:
    jt_model = jt_model.withColumn(c, F.col(c).cast("double"))

jt_model = (
    jt_model
    .filter(F.col(target_col).isNotNull())
    .fillna(0)
)

print("After cast/fill:")
jt_model.printSchema()



In [ ]:
# =========================================================
# 5. Train / test split by year
# =========================================================
train_df = jt_model.filter(F.col("jt_year") == 2012)
test_df  = jt_model.filter(F.col("jt_year") == 2013)

print("Train rows before balance:", train_df.count())
print("Test rows before balance :", test_df.count())

train_df.groupBy(target_col).count().orderBy(target_col).show()
test_df.groupBy(target_col).count().orderBy(target_col).show()

# =========================================================
# 6. Simple undersampling on training set
# =========================================================
train_pos = train_df.filter(F.col(target_col) == 1.0)
train_neg = train_df.filter(F.col(target_col) == 0.0)

pos_count = train_pos.count()
neg_count = train_neg.count()

print("Positive count:", pos_count)
print("Negative count:", neg_count)

if pos_count == 0 or neg_count == 0:
    raise ValueError("Train set only has one class. Cannot train classifier.")

if pos_count < neg_count:
    frac = pos_count / neg_count
    train_neg_sample = train_neg.sample(withReplacement=False, fraction=frac, seed=42)
    train_balanced = train_pos.unionByName(train_neg_sample)
else:
    frac = neg_count / pos_count
    train_pos_sample = train_pos.sample(withReplacement=False, fraction=frac, seed=42)
    train_balanced = train_pos_sample.unionByName(train_neg)

train_balanced = train_balanced.orderBy(F.rand(42))

print("Train rows after balance:", train_balanced.count())
train_balanced.groupBy(target_col).count().orderBy(target_col).show()

# =========================================================
# 7. Final feature list
# =========================================================
# jt_year 不参与训练
final_feature_cols = [c for c in jt_model.columns if c not in [target_col, "jt_year"]]

print("Number of final features:", len(final_feature_cols))
print("First 20 final features:", final_feature_cols[:20])



In [ ]:
# =========================================================
# 8. Assemble + Random Forest
# =========================================================
assembler = VectorAssembler(
    inputCols=final_feature_cols,
    outputCol="features",
    handleInvalid="skip"
)

rf = RandomForestClassifier(
    labelCol=target_col,
    featuresCol="features",
    numTrees=50,
    maxDepth=8,
    seed=42
)

pipeline = Pipeline(stages=[assembler, rf])

# =========================================================
# 9. Train
# =========================================================
rf_model = pipeline.fit(train_balanced)



In [ ]:
# =========================================================
# 10. Predict
# =========================================================
pred_df = rf_model.transform(test_df)

print("Prediction rows:", pred_df.count())

pred_df.select(target_col, "prediction", "probability").show(20, truncate=False)



In [ ]:
# =========================================================
# 11. Evaluate
# =========================================================
auc_eval = BinaryClassificationEvaluator(
    labelCol=target_col,
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)

pr_eval = BinaryClassificationEvaluator(
    labelCol=target_col,
    rawPredictionCol="rawPrediction",
    metricName="areaUnderPR"
)

acc_eval = MulticlassClassificationEvaluator(
    labelCol=target_col,
    predictionCol="prediction",
    metricName="accuracy"
)

f1_eval = MulticlassClassificationEvaluator(
    labelCol=target_col,
    predictionCol="prediction",
    metricName="f1"
)

auc = auc_eval.evaluate(pred_df)
aupr = pr_eval.evaluate(pred_df)
acc = acc_eval.evaluate(pred_df)
f1 = f1_eval.evaluate(pred_df)

print("AUC  =", auc)
print("AUPR =", aupr)
print("ACC  =", acc)
print("F1   =", f1)

# =========================================================
# 12. Confusion matrix
# =========================================================
print("Confusion matrix:")
pred_df.groupBy(target_col, "prediction").count().orderBy(target_col, "prediction").show()

# =========================================================
# 13. Feature importance
# =========================================================
rf_stage = rf_model.stages[-1]
importances = rf_stage.featureImportances

feat_imp = list(zip(final_feature_cols, importances.toArray().tolist()))
feat_imp_sorted = sorted(feat_imp, key=lambda x: x[1], reverse=True)

print("Top 20 feature importances:")
for feat, score in feat_imp_sorted[:20]:
    print(f"{feat:40s} {score:.6f}")